# **PRE PROCESSING**

In [2]:
import pandas as pd
import re

In [ ]:
nabil = pd.read_csv("final_tamil_english_pos_aligned.csv")

In [ ]:
nabil.columns

Index(['Tamil', 'English', 'Label'], dtype='object')

In [ ]:
nabil['Label'].value_counts()

Label
NOUN     5608
ADP      2486
PUNCT    2117
VERB     1833
PROPN    1777
NUM      1483
ADJ      1430
DET      1420
AUX       726
ADV       479
PRON      410
CCONJ     364
PART      244
SCONJ     164
SYM       100
X          16
INTJ       12
Name: count, dtype: int64

In [ ]:
df = pd.read_csv("dictionary.csv")

In [ ]:
# Define POS mapping
pos_map = {
    "n": "noun",
    "v": "verb",
    "adv": "adverb",
    "a": "adjective"
}

In [ ]:
df.head(4)

,tamil,eng
0,"n. நெடுங்கணக்கு, தொடக்கச்சுவடி, அடிப்படைக்கருத...",A B C
1,"a.adv இரண்டிற்கான, இரண்டினிடையே.",a dcux
2,"adv. முற்ற, முழுக்க, அடிவரையில்.",a fond
3,adv. மேலும் வலிய காரணத்தால்.,a fortiori


In [ ]:
# Function to parse each Tamil entry
def parse_tamil_entry(tamil_entry):
    tamil_entry = str(tamil_entry).strip()

    # Regex to extract POS at start (like n., v., adj., a.adv, adj.adv)
    pos_match = re.match(r'^([a-z\.]+)\s+', tamil_entry)
    if pos_match:
        raw_pos = pos_match.group(1)  # e.g., 'n', 'a.adv'
        tamil_words_part = tamil_entry[pos_match.end():]  # rest after POS
    else:
        raw_pos = None
        tamil_words_part = tamil_entry

    # Process POS
    if raw_pos:
        pos_list = []
        for p in raw_pos.split('.'):
            if p in pos_map:
                pos_list.append(pos_map[p])
        pos_final = ",".join(pos_list) if pos_list else "none"
    else:
        pos_final = "none"

    # Remove trailing periods
    tamil_words_part = tamil_words_part.rstrip('.')

    # Split Tamil words by comma
    tamil_words = [w.strip() for w in tamil_words_part.split(',') if w.strip()]

    return [(word, pos_final) for word in tamil_words]

In [ ]:
# Prepare list of rows
rows = []
for idx, row in df.iterrows():
    eng_word = row['eng']
    tamil_entry = row['tamil']
    parsed_words = parse_tamil_entry(tamil_entry)
    for tamil_word, pos in parsed_words:
        rows.append({'tamil_word': tamil_word, 'eng_word': eng_word, 'pos': pos})

In [ ]:
# Create DataFrame
refined_df = pd.DataFrame(rows)

In [ ]:
# Show first 20 rows to check
refined_df.head(20)

,tamil_word,eng_word,pos
0,நெடுங்கணக்கு,A B C,noun
1,தொடக்கச்சுவடி,A B C,noun
2,அடிப்படைக்கருத்து,A B C,noun
3,இரண்டிற்கான,a dcux,"adjective,adverb"
4,இரண்டினிடையே,a dcux,"adjective,adverb"
5,முற்ற,a fond,adverb
6,முழுக்க,a fond,adverb
7,அடிவரையில்,a fond,adverb
8,மேலும் வலிய காரணத்தால்,a fortiori,adverb
9,கதவை மூடிக்கொண்டு,a huis clos,adverb


In [ ]:
refined_df.shape

(195851, 3)

In [ ]:
# Path to save
save_path = r"D:\COLLEGE\SEM 5\MLM\refined_dictionary.csv"

# Save to CSV without the index column
refined_df.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"Refined dictionary saved to: {save_path}")

Refined dictionary saved to: D:\COLLEGE\SEM 5\MLM\refined_dictionary.csv


In [ ]:
mask = refined_df['pos'].str.contains(r'-\d+\s*-\d+', regex=True) | \
       refined_df['eng_word'].str.contains(r'-\d+\s*-\d+', regex=True)

# Show all rows where the pattern is present
problematic_rows = refined_df[mask]

print("Rows containing '-2 -2' patterns:")
print(problematic_rows)

Rows containing '-2 -2' patterns:
Empty DataFrame
Columns: [tamil_word, eng_word, pos]
Index: []


In [ ]:
pattern = r'^-\d+'

# Create a boolean mask for rows that match the pattern
mask = refined_df['tamil_word'].str.contains(pattern, regex=True)

# Filter the DataFrame to see only the problematic rows
problematic_rows = refined_df[mask]

# Show count and the rows
print(f"Number of rows with leading '-number': {problematic_rows.shape[0]}")
print(problematic_rows)

Number of rows with leading '-number': 0
Empty DataFrame
Columns: [tamil_word, eng_word, pos]
Index: []


In [ ]:
# Remove leading '-2', '-1', etc., and any whitespace following them
refined_df['tamil_word'] = refined_df['tamil_word'].str.replace(r'^(-\d+\s*)+', '', regex=True)

# Optional: strip extra whitespace
refined_df['tamil_word'] = refined_df['tamil_word'].str.strip()

In [ ]:
# Path to save
save_path = r"D:\COLLEGE\SEM 5\MLM\refined_dictionary.csv"

# Save to CSV without the index column
refined_df.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"Refined dictionary saved to: {save_path}")

Refined dictionary saved to: D:\COLLEGE\SEM 5\MLM\refined_dictionary.csv


# Building Tamil - eng Dictionary

In [ ]:
tamil_to_english = pd.Series(refined_df.eng_word.values, index=refined_df.tamil_word).to_dict()

In [ ]:
tamil_sentence = "அவள் பெரிய சிவப்பு கார் ஓட்டுகிறாள்"
words = tamil_sentence.split()  # crude tokenization by space
translated_words = [tamil_to_english.get(w, w) for w in words]  # fallback: keep original if not found
english_sentence = " ".join(translated_words)
print(english_sentence)

அவள் unco redness கார் ஓட்டுகிறாள்


# Combine Wordnet and Refined dict

In [ ]:
df1 = pd.read_csv(r"D:\COLLEGE\SEM 5\MLM\refined_dictionary.csv")

In [ ]:
df2 = pd.read_csv(r"D:\COLLEGE\SEM 5\MLM\wordNet.csv")

In [ ]:
df1.columns

Index(['tamil_word', 'eng_word', 'pos'], dtype='object')

In [ ]:
df1 = df1.drop('tamil_word',axis=1)

In [ ]:
df1.columns

Index(['eng_word', 'pos'], dtype='object')

In [ ]:
df1.drop_duplicates(inplace=True)

In [ ]:
df1.shape

(55013, 2)

In [ ]:
df1['pos'].value_counts()

pos
noun                34840
adjective           10661
none                 5079
verb                 3448
adverb                984
adjective,adverb        1
Name: count, dtype: int64

In [ ]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 55013 entries, 0 to 195850
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   eng_word  55012 non-null  object
 1   pos       55013 non-null  object
dtypes: object(2)
memory usage: 1.3+ MB


In [ ]:
df11 = df1[df1['pos']!='none']

In [ ]:
df11.shape

(49934, 2)

In [ ]:
df11['pos'].value_counts()

pos
noun                34840
adjective           10661
verb                 3448
adverb                984
adjective,adverb        1
Name: count, dtype: int64

In [ ]:
df1 = df11

In [ ]:
df1.columns

Index(['eng_word', 'pos'], dtype='object')

In [ ]:
df2.columns

Index(['word', 'pos', 'definition', 'examples'], dtype='object')

In [ ]:
df2.rename(columns={'word': 'eng_word'}, inplace=True)

In [ ]:
df2.columns

Index(['eng_word', 'pos', 'definition', 'examples'], dtype='object')

In [ ]:
df2.drop(['definition','examples'],inplace=True,axis=1)

In [ ]:
df2.columns

Index(['eng_word', 'pos'], dtype='object')

In [ ]:
df2['pos'].value_counts()

pos
n    146347
v     25047
s     20336
a      9668
r      5580
Name: count, dtype: int64

In [ ]:
df1['pos'].value_counts()

pos
noun                34840
adjective           10661
verb                 3448
adverb                984
adjective,adverb        1
Name: count, dtype: int64

In [ ]:
df2['pos'] = df2['pos'].replace({
    'n': 'noun',
    'v': 'verb',
    's': 'adjective',
    'a': 'adjective',
    'r': 'adverb'
})

In [ ]:
df2['pos'].value_counts()

pos
noun         146347
adjective     30004
verb          25047
adverb         5580
Name: count, dtype: int64

In [ ]:
combined_df = pd.concat([df1,df2],ignore_index=True)

In [ ]:
combined_df

,eng_word,pos
0,A B C,noun
1,a dcux,"adjective,adverb"
2,a fond,adverb
3,a fortiori,adverb
4,a huis clos,adverb
...,...,...
256907,fog_up,verb
256908,char,verb
256909,coal,verb
256910,haze,verb


In [ ]:
combined_df.drop_duplicates(inplace=True)

In [ ]:
combined_df.shape

(180343, 2)

In [ ]:
combined_df['pos'].value_counts()

pos
noun                136276
adjective            26477
verb                 12636
adverb                4953
adjective,adverb         1
Name: count, dtype: int64

In [ ]:
combined_df.columns

Index(['eng_word', 'pos'], dtype='object')

In [ ]:
# Path to save
save_path = r"D:\COLLEGE\SEM 5\MLM\English_word_POS_TAG.csv"

# Save to CSV without the index column
combined_df.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"Combined dictionary saved to: {save_path}")

Refined dictionary saved to: D:\COLLEGE\SEM 5\MLM\English_word_POS_TAG.csv


# ---------------------------- START ------------------------

# POS MODEL TRAINING

In [3]:
df1 = pd.read_csv(r"english_pos_tag_merged_nabil.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'english_pos_tag_merged_nabil.csv'

In [ ]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 181141 entries, 0 to 181140
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   eng_word  181139 non-null  object
 1   pos       181141 non-null  object
dtypes: object(2)
memory usage: 2.8+ MB


In [ ]:
df1.columns

Index(['eng_word', 'pos'], dtype='object')

In [ ]:
df1['pos'].value_counts()

pos
noun                136675
adjective            26523
verb                 12847
adverb                4958
adposition              34
pronoun                 27
conjunction             21
auxiliary               17
numeral                 16
determiner              16
particle                 3
other                    2
interjection             1
adjective,adverb         1
Name: count, dtype: int64

In [ ]:
df1.shape

(181141, 2)

In [ ]:
df1 = df1[~df1['pos'].isin(['particle', 'other', 'interjection'])]

In [ ]:
df1['pos'].value_counts()

pos
noun                136675
adjective            26523
verb                 12847
adverb                4958
adposition              34
pronoun                 27
conjunction             21
auxiliary               17
determiner              16
numeral                 16
adjective,adverb         1
Name: count, dtype: int64

# Preprocessing

In [ ]:
df1.drop_duplicates(inplace=True)

In [ ]:
df1.shape

(181135, 2)

In [ ]:
df1.isnull().sum()

eng_word    2
pos         0
dtype: int64

In [ ]:
df1 = df1.dropna()

In [ ]:
df1.isnull().sum()

eng_word    0
pos         0
dtype: int64

In [ ]:
# Normalize POS tags
df1['pos'] = df1['pos'].replace({
    'noun': 'noun',
    'verb': 'verb',
    'adjective': 'adjective',
    'adverb': 'adverb',
    'adjective,adverb': 'adjective'   # merge rare tag
})

In [ ]:
df1['pos'].value_counts()

pos
noun           136674
adjective       26523
verb            12847
adverb           4958
adposition         34
pronoun            27
conjunction        21
auxiliary          17
numeral            16
determiner         16
Name: count, dtype: int64

**FEATURE EXTRACTION**

In [ ]:
def add_features(df):
    df['word_len'] = df['eng_word'].apply(len)
    df['prefix_2'] = df['eng_word'].str[:2]
    df['prefix_3'] = df['eng_word'].str[:3]
    df['suffix_2'] = df['eng_word'].str[-2:]
    df['suffix_3'] = df['eng_word'].str[-3:]
    return df

df1 = add_features(df1)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 4))
X_tfidf = vectorizer.fit_transform(df1['eng_word'])

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack

ohe = OneHotEncoder(handle_unknown='ignore')

# Fit-transform the prefix/suffix categorical features
X_cat = ohe.fit_transform(df1[['prefix_2', 'prefix_3', 'suffix_2', 'suffix_3']])

# Numeric features (word_len)
X_num = df1[['word_len']].values

# Combine with your TF-IDF vector
X = hstack([X_tfidf, X_num, X_cat])
y = df1['pos']

# Model Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42,stratify=y_encoded)

In [ ]:
model = LogisticRegression(max_iter=1000, n_jobs=-1,class_weight='balanced')
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [ ]:
# with prefix,suffix,stratify = y
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("Overall accuracy:", accuracy_score(y_test, y_pred))

              precision    recall  f1-score   support

   adjective       0.67      0.81      0.73      5295
      adverb       0.60      0.82      0.69       991
        noun       0.97      0.83      0.90     27256
        verb       0.43      0.80      0.56      2527

    accuracy                           0.83     36069
   macro avg       0.67      0.82      0.72     36069
weighted avg       0.87      0.83      0.84     36069

Overall accuracy: 0.8280795142643267


In [ ]:
# Evaluate(without prefix,suffix,..)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))

              precision    recall  f1-score   support

   adjective       0.82      0.62      0.70      5281
      adverb       0.90      0.65      0.75       969
        noun       0.88      0.97      0.92     27292
        verb       0.71      0.38      0.49      2527

    accuracy                           0.87     36069
   macro avg       0.83      0.65      0.72     36069
weighted avg       0.86      0.87      0.86     36069



# Hyper paramter tuning

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
# -------------------------------------
# Step 2: Encode target
# -------------------------------------
le = LabelEncoder()
y = le.fit_transform(df1['pos'])

# -------------------------------------
# Step 3: Define preprocessing
# -------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2)), 'eng_word'),
        ('num', StandardScaler(with_mean=False), ['word_len']),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True),
         ['prefix_2', 'prefix_3', 'suffix_2', 'suffix_3'])
    ],
    sparse_threshold=0.3
)

# -------------------------------------
# Step 4: Define pipeline
# -------------------------------------
pipe = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('lr', LogisticRegression(
        solver='liblinear',
        class_weight='balanced',
        n_jobs=-1,
        max_iter=500
    ))
])

In [ ]:
# -------------------------------------
# Step 5: Split data
# -------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    df1, y, test_size=0.2, random_state=42,stratify = y
)

In [ ]:
# -------------------------------------
# Step 6: Minimal Grid Search
# -------------------------------------
param_grid = {
    'lr__C': [0.1, 1, 10],         # regularization strength
    'lr__penalty': ['l1', 'l2']    # both supported by liblinear
}

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

Fitting 3 folds for each of 6 candidates, totalling 18 fits


C:\Users\sanka\OneDrive\Documents\Anancoda\envs\mp_env\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
C:\Users\sanka\OneDrive\Documents\Anancoda\envs\mp_env\lib\site-packages\sklearn\linear_model\_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 12.
  warnings.warn(


,estimator,Pipeline(step...liblinear'))])
,param_grid,"{'lr__C': [0.1, 1, ...], 'lr__penalty': ['l1', 'l2']}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,3
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('tfidf', ...), ('num', ...), ...]"


In [ ]:
# -------------------------------------
# Step 7: Evaluate best model
# -------------------------------------
print("\nBest Parameters:", grid.best_params_)
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=le.classes_))
print("Overall accuracy:", accuracy_score(y_test, y_pred))


Best Parameters: {'lr__C': 1, 'lr__penalty': 'l1'}

Classification Report:
               precision    recall  f1-score   support

   adjective       0.69      0.73      0.71      5305
  adposition       0.00      0.00      0.00         7
      adverb       0.64      0.81      0.71       992
   auxiliary       0.00      0.00      0.00         3
 conjunction       0.00      0.00      0.00         4
  determiner       0.00      0.00      0.00         3
        noun       0.94      0.89      0.91     27335
     numeral       0.00      0.00      0.00         3
     pronoun       0.11      0.20      0.14         5
        verb       0.51      0.67      0.58      2570

    accuracy                           0.85     36227
   macro avg       0.29      0.33      0.31     36227
weighted avg       0.86      0.85      0.85     36227

Overall accuracy: 0.8461920666905899


In [ ]:
# -------------------------------------
# Save trained pipeline and label encoder
# -------------------------------------
import joblib

# Save the best model (includes preprocessing + logistic regression)
joblib.dump(best_model, "best_pipeline.pkl")

# Save the label encoder
joblib.dump(le, "label_encoder.pkl")

print("✅ Model and label encoder saved successfully.")


✅ Model and label encoder saved successfully.


In [ ]:
# -------------------------------------
# Load saved model and make predictions
# -------------------------------------
import joblib

# Load pipeline and label encoder
loaded_pipeline = joblib.load("best_pipeline.pkl")
le = joblib.load("label_encoder.pkl")

# Example new data (same column names as training data)
new_data = [
    {
        'eng_word': 'food',
        'word_len': 4,
        'prefix_2': 'fo',
        'prefix_3': 'foo',
        'suffix_2': 'od',
        'suffix_3': 'ood'
    },
    {
        'eng_word': 'handsome',
        'word_len': 8,
        'prefix_2': 'ha',
        'prefix_3': 'han',
        'suffix_2': 'me',
        'suffix_3': 'ome'
    }
]

import pandas as pd
new_df = pd.DataFrame(new_data)

# Predict POS tags
y_pred = loaded_pipeline.predict(new_df)

# Decode labels back to original POS names
predicted_pos = le.inverse_transform(y_pred)

for word, pos_tag in zip(new_df['eng_word'], predicted_pos):
    print(f"{word:10s} → {pos_tag}")


food       → noun
handsome   → adjective


# END -----------------

# Pipe line

In [ ]:
# -------------------------------------
# Step 2: Encode target
# -------------------------------------
le = LabelEncoder()
y = le.fit_transform(df1['pos'])

# -------------------------------------
# Step 3: Define preprocessing
# -------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2)), 'eng_word'),
        ('num', StandardScaler(with_mean=False), ['word_len']),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True),
         ['prefix_2', 'prefix_3', 'suffix_2', 'suffix_3'])
    ],
    sparse_threshold=0.3  # ensures output remains sparse
)

# -------------------------------------
# Step 4: Define full pipeline
# -------------------------------------
pipe = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('lr', LogisticRegression(
        solver='liblinear',
        class_weight='balanced',
        max_iter=500,
        n_jobs=-1
    ))
])

In [ ]:
# -------------------------------------
# Step 5: Split data
# -------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    df1, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# -------------------------------------
# Step 6: Train model
# -------------------------------------
pipe.fit(X_train, y_train)

C:\Users\sanka\OneDrive\Documents\Anancoda\envs\mp_env\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
C:\Users\sanka\OneDrive\Documents\Anancoda\envs\mp_env\lib\site-packages\sklearn\linear_model\_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 12.
  warnings.warn(


,steps,"[('preprocess', ...), ('lr', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('tfidf', ...), ('num', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
# -------------------------------------
# Step 7: Evaluate
# -------------------------------------
y_pred = pipe.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("Overall accuracy:", accuracy_score(y_test, y_pred))

              precision    recall  f1-score   support

   adjective       0.69      0.73      0.71      5305
  adposition       0.00      0.00      0.00         7
      adverb       0.65      0.80      0.72       992
   auxiliary       0.12      0.33      0.18         3
 conjunction       0.00      0.00      0.00         4
  determiner       0.00      0.00      0.00         3
        noun       0.93      0.89      0.91     27335
     numeral       0.00      0.00      0.00         3
     pronoun       0.15      0.40      0.22         5
        verb       0.50      0.64      0.56      2570

    accuracy                           0.84     36227
   macro avg       0.30      0.38      0.33     36227
weighted avg       0.86      0.84      0.85     36227

Overall accuracy: 0.8443150136638419


# Save and Test the model

In [ ]:
import joblib

joblib.dump(model, 'pos_predictor_model.pkl')
joblib.dump(vectorizer, 'pos_vectorizer.pkl')
joblib.dump(le, 'pos_label_encoder.pkl')

In [ ]:
# Load
model = joblib.load('pos_predictor_model.pkl')
vectorizer = joblib.load('pos_vectorizer.pkl')
le = joblib.load('pos_label_encoder.pkl')

def predict_pos(word):
    df_temp = pd.DataFrame({'eng_word': [word]})
    df_temp = add_features(df_temp)

    X_tfidf = vectorizer.transform(df_temp['eng_word'])
    X_numeric = df_temp[['word_len']].values
    X = hstack([X_tfidf, X_numeric])

    pred = model.predict(X)
    return le.inverse_transform(pred)[0]

print(predict_pos("beautiful"))
print(predict_pos("quickly"))
print(predict_pos("run"))

adjective
adverb
noun


In [ ]:
df1[df1['eng_word'] == 'run']

,eng_word,pos,word_len,prefix_2,prefix_3,suffix_2,suffix_3
34925,run,noun,3,ru,run,un,run
172866,run,verb,3,ru,run,un,run


# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import time

In [ ]:
start = time.time()

model = RandomForestClassifier(
    n_estimators=100,        # fewer trees
    max_depth=25,            # limit depth (prevents overfitting & speeds up)
    min_samples_split=10,    # don’t split small nodes
    min_samples_leaf=5,      # make leaves bigger
    max_features='sqrt',     # use only subset of features per split
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

RandomForestClassifier(max_depth=25, min_samples_leaf=5, min_samples_split=10,
                       n_jobs=-1, random_state=42)

In [ ]:
print("Training time:", round(time.time() - start, 2), "seconds")

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))

Training time: 11.33 seconds
              precision    recall  f1-score   support

   adjective       1.00      0.00      0.00      5281
      adverb       1.00      0.03      0.06       969
        noun       0.76      1.00      0.86     27292
        verb       0.00      0.00      0.00      2527

    accuracy                           0.76     36069
   macro avg       0.69      0.26      0.23     36069
weighted avg       0.75      0.76      0.65     36069



C:\Users\sanka\OneDrive\Documents\Anancoda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\sanka\OneDrive\Documents\Anancoda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\sanka\OneDrive\Documents\Anancoda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is",

# Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score
import joblib
import time

In [ ]:
# Start timer
start = time.time()

# Train Naive Bayes model
model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

In [ ]:
# Predict
y_pred = model.predict(X_test)

# End timer
print(f"Training time: {time.time() - start:.2f} seconds")

# Evaluate performance
print("\nModel Performance:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("Overall Accuracy:", accuracy_score(y_test, y_pred))

Training time: 10.58 seconds

Model Performance:

              precision    recall  f1-score   support

   adjective       0.72      0.11      0.20      5281
      adverb       0.00      0.00      0.00       969
        noun       0.77      1.00      0.87     27292
        verb       0.94      0.02      0.04      2527

    accuracy                           0.77     36069
   macro avg       0.61      0.28      0.28     36069
weighted avg       0.76      0.77      0.69     36069

Overall Accuracy: 0.7711885552690676


C:\Users\sanka\OneDrive\Documents\Anancoda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\sanka\OneDrive\Documents\Anancoda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\sanka\OneDrive\Documents\Anancoda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is",

In [ ]:
# Save model for future use
joblib.dump(model, "pos_naive_bayes_model.pkl")
print("\nModel saved as 'pos_naive_bayes_model.pkl'")

# SVM

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
import time

In [ ]:
# Train RBF SVM
start = time.time()
svm_model = SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced')
svm_model.fit(X_train, y_train)
end = time.time()

In [ ]:
# Evaluate
y_pred = svm_model.predict(X_test)

print(f"Training time: {end - start:.2f} seconds\n")
print("Model Performance:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("Overall Accuracy:", accuracy_score(y_test, y_pred))

Training time: 9285.86 seconds

Model Performance:

              precision    recall  f1-score   support

   adjective       0.63      0.78      0.70      5281
      adverb       0.59      0.80      0.68       969
        noun       0.97      0.76      0.85     27292
        verb       0.32      0.85      0.47      2527

    accuracy                           0.77     36069
   macro avg       0.63      0.80      0.67     36069
weighted avg       0.86      0.77      0.80     36069

Overall Accuracy: 0.7708004103246555


# Sentence correction

In [ ]:
!pip install transformers torch gector

ERROR: Could not find a version that satisfies the requirement gector (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for gector


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
# ---------- 1️⃣ T5-based Model ----------
t5_model_name = t5_model_name = "prithivida/grammar_error_correcter_v1"
t5_tokenizer = AutoTokenizer.from_pretrained(t5_model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

C:\Users\sanka\OneDrive\Documents\Anancoda\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sanka\.cache\huggingface\hub\models--prithivida--grammar_error_correcter_v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [ ]:
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_model_name)

In [ ]:
def correct_t5(sentence):
    input_text = "gec: " + sentence
    input_ids = t5_tokenizer.encode(input_text, return_tensors="pt", truncation=True)
    outputs = t5_model.generate(input_ids, max_length=64, num_beams=5, early_stopping=True)
    return t5_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# ---------- 2️⃣ GECToR Model ----------
from gector.gec_model import GecBERTModel

pip install --upgrade --no-cache-dir git+https://github.com/grammarly/gector.git

# You can use 'roberta' or 'bert' backbone
gec_model = GecBERTModel(
    vocab_path=None,
    model_paths=["roberta-large"],
    model_name="roberta",
    special_tokens_fix=0,
    log=False
)

def correct_gector(sentence):
    preds, _ = gec_model.handle_batch([sentence])
    return preds[0]

**SENTENCES**

# ---------- 3️⃣ Test Sentences ----------
sentences = [
    "i go a school",
    "she not like mango",
    "they playing football yesterday"
]

print("🔍 Grammar Correction Comparison\n")
for s in sentences:
    #t5_corrected = correct_t5(s)
    gector_corrected = correct_gector(s)
    print(f"Original: {s}")
    print(f"T5-based correction: {t5_corrected}")
    print(f"GECToR correction   : {gector_corrected}\n")

In [ ]:
!pip install happytransformer

  Using cached happytransformer-3.0.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached torch-2.9.0-cp310-cp310-win_amd64.whl.metadata (30 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
  Using cached datasets-2.21.0-py3-none-any.whl.metadata (21 kB)
  Using cached accelerate-0.34.2-py3-none-any.whl.metadata (19 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
  Using cached wandb-0.22.3-py3-none-win_amd64.whl.metadata (10 kB)
  Using cached huggingface_hub-1.1.2-py3-none-any.whl.metadata (13 kB)
  Using cached safetensors-0.6.2-cp38-abi3-win_amd64.whl.metadata (4.1 kB)
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached pyarrow-22.0.0-cp310-cp310-win_amd64.whl.metadata (3.3 kB)
  Using cached dill-0.3.8-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-3.6.0-cp310-cp310-win_amd64.whl.metadata (13 kB)
  Using cached multiprocess-0.70.18

In [ ]:
from happytransformer import HappyTextToText

C:\Users\sanka\OneDrive\Documents\Anancoda\envs\mp_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load a pre-trained grammar correction model
happy_tt = HappyTextToText("T5", "vennify/t5-base-grammar-correction")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
11/07/2025 12:46:50 - INFO - happytransformer.happy_transformer -   Using device: cpu
C:\Users\sanka\OneDrive\Documents\Anancoda\envs\mp_env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sanka\.cache\huggingface\hub\models--vennify--t5-base-grammar-correction. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need

In [ ]:
# Input text
text = "i a boy "
result = happy_tt.generate_text(text)
print(result.text)

Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I am a boy. I am a boy.


In [ ]:
import pandas as pd

# Merge Tamil dict

In [ ]:
df1 = pd.read_csv(r"tamil_english_clean_with_POS_without_duplicates.csv")

In [ ]:
df1.shape

(3132, 3)

In [ ]:
df11.columns

Index(['english', 'label'], dtype='object')

In [ ]:
df2.shape

(180343, 2)

In [ ]:
df2 = pd.read_csv(r"D:\COLLEGE\SEM 5\MLM\English_word_POS_TAG.csv")

In [ ]:
df2.columns

Index(['eng_word', 'pos'], dtype='object')

In [ ]:
df11 = df1[ ['english','label'] ]

In [ ]:
df11['label'].value_counts()

label
NOUN     1380
VERB      458
PROPN     344
ADJ       334
ADP       170
ADV       113
PRON       88
DET        72
AUX        58
SCONJ      40
NUM        34
PART       22
CCONJ      16
X           2
INTJ        1
Name: count, dtype: int64

In [ ]:
df2['pos'].value_counts()

pos
noun                136276
adjective            26477
verb                 12636
adverb                4953
adjective,adverb         1
Name: count, dtype: int64

In [ ]:
# Define mapping from df11 label → normalized form
label_map = {
    'NOUN': 'noun',
    'PROPN': 'noun',
    'VERB': 'verb',
    'ADJ': 'adjective',
    'ADV': 'adverb',
    'ADP': 'adposition',
    'PRON': 'pronoun',
    'DET': 'determiner',
    'AUX': 'auxiliary',
    'SCONJ': 'conjunction',
    'CCONJ': 'conjunction',
    'NUM': 'numeral',
    'PART': 'particle',
    'INTJ': 'interjection',
    'X': 'other'
}

# Apply mapping to df11
df11['label_normalized'] = df11['label'].map(label_map).fillna(df11['label'].str.lower())

print(df11['label_normalized'].value_counts())

label_normalized
noun            1724
verb             458
adjective        334
adposition       170
adverb           113
pronoun           88
determiner        72
auxiliary         58
conjunction       56
numeral           34
particle          22
other              2
interjection       1
Name: count, dtype: int64


In [ ]:
df11

,english,label,label_normalized
0,achieving,VERB,verb
1,growth,NOUN,noun
2,difficult,ADJ,adjective
3,economic,ADJ,adjective
4,these,DET,determiner
...,...,...,...
3127,term,NOUN,noun
3128,loans,NOUN,noun
3129,paid,VERB,verb
3130,may,AUX,auxiliary


In [ ]:
df2.columns

Index(['eng_word', 'pos'], dtype='object')

In [ ]:
# ✅ Step 2: Rename columns to match df2
df11_renamed = df11.rename(columns={'english': 'eng_word', 'label_normalized': 'pos'})
df2_renamed = df2.rename(columns={'pos': 'pos'})

# ✅ Step 3: Concatenate vertically (stack rows)
merged_df = pd.concat(
    [df11_renamed[['eng_word', 'pos']], df2_renamed[['eng_word', 'pos']]],
    axis=0,
    ignore_index=True
)

In [ ]:
merged_df

,eng_word,pos
0,achieving,verb
1,growth,noun
2,difficult,adjective
3,economic,adjective
4,these,determiner
...,...,...
183470,shade_off,verb
183471,cloud_up,verb
183472,blight,verb
183473,run_dry,verb


In [ ]:
merged_df.drop_duplicates(inplace=True)

In [ ]:
merged_df.shape

(181141, 2)

In [ ]:
# ✅ Step 4: Save merged dataframe
merged_df.to_csv("english_pos_tag_merged_nabil.csv", index=False)

print("✅ Vertically concatenated DataFrame saved as 'english_pos_tag_merged_nabil.csv'")
print("Shape:", merged_df.shape)
print(merged_df.head())

✅ Vertically concatenated DataFrame saved as 'english_pos_tag_merged_nabil.csv'
Shape: (181141, 2)
    eng_word         pos
0  achieving        verb
1     growth        noun
2  difficult   adjective
3   economic   adjective
4      these  determiner


In [ ]:
merged_df

,eng_word,pos
0,achieving,verb
1,growth,noun
2,difficult,adjective
3,economic,adjective
4,these,determiner
...,...,...
183470,shade_off,verb
183471,cloud_up,verb
183472,blight,verb
183473,run_dry,verb


In [ ]:
# ✅ Step 4: Save final dataframe
merged_df.to_csv("english_pos_tag_merged_nabil.csv", index=False)

print("✅ Horizontally concatenated DataFrame saved as 'english_pos_tag_merged_nabil.csv'")
print(merged_df.head())
print("\nShape:", merged_df.shape)

# Parallel

In [ ]:
import os
import pandas as pd
import glob

# Path
save_dir = r"D:\COLLEGE\SEM 5\MLM\TAMIL-ENGLISH"
os.makedirs(save_dir, exist_ok=True)  # Create if it doesn't exist

# Merge Tamil files
tamil_lines = []
for fname in sorted(glob.glob(os.path.join(save_dir, "data.ta*"))):
    with open(fname, encoding="utf-8") as infile:
        tamil_lines.extend(infile.readlines())

# Merge English files
english_lines = []
for fname in sorted(glob.glob(os.path.join(save_dir, "data.en*"))):
    with open(fname, encoding="utf-8") as infile:
        english_lines.extend(infile.readlines())

# Equalize line count
min_len = min(len(tamil_lines), len(english_lines))
tamil_lines = tamil_lines[:min_len]
english_lines = english_lines[:min_len]

# Create dataframe
df = pd.DataFrame({
    "tamil_sentence": [line.strip() for line in tamil_lines],
    "english_sentence": [line.strip() for line in english_lines]
})

# Save as CSV
output_path = os.path.join(save_dir, "tamil_english_sentences.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✅ Saved merged dataset as:\n{output_path}")
print("Rows:", len(df))
print(df.head())


✅ Saved merged dataset as:
D:\COLLEGE\SEM 5\MLM\TAMIL-ENGLISH\tamil_english_sentences.csv
Rows: 289451
                                      tamil_sentence  \
0  ராஜாவாகிய ஆகாஸ் அரசாளும்போது தம்முடைய பாதகத்தி...   
1  சர்வதேச நாணய நிதியம் இலங்கைக்கு கடன் வழங்கினால...   
2  தற்போது அதற்கு எதிராக வாதாடுகிறார் சர்வதேச சட...   
3  அமெரிக்காவின் மூன்றாம் பெரிய கார் தயாரிப்பு நி...   
4  மேலும் இனைவிட்டு தலிபானால் வெளியேற்றப்பட்ட 199...   

                                    english_sentence  
0  moreover all the vessels , which king ahaz in ...  
1  similar conditions will be imposed if the sri ...  
2  now kornelius argues the opposite instead of e...  
3  chrysler the third largest us automaker filed ...  
4  moreover , khan has been in exile in iran for ...  
